# Phase 1 — Lesson 07: Bayes' Theorem
> Faithful study notes built directly from the lesson source. Concepts, code, diagrams, and exercises.

**Source:** ai-engineering-from-scratch -> Phase 1 (Math Foundations), Lesson 07: Bayes' Theorem
**Type:** Build | **Languages:** Python | **Time:** ~75 minutes
**Prerequisites:** Phase 1, Lesson 06 (Probability Fundamentals)



# What You'll Learn

---

## Learning Objectives

- Apply Bayes' theorem to compute posterior probabilities from priors, likelihoods, and evidence
- Build a Naive Bayes text classifier from scratch with Laplace smoothing and log-space computation
- Compare MLE and MAP estimation and explain how MAP corresponds to L2 regularization
- Implement sequential Bayesian updating using Beta-Binomial conjugate priors for A/B testing


# The Problem

---

A medical test is 99% accurate. You test positive. What are the chances you actually have the disease?

Most people say 99%. The real answer depends on how rare the disease is. If 1 in 10,000 people have it, a positive result only gives you about a 1% chance of being sick.


# Bayes' Theorem

---

```
P(A|B) = P(B|A) × P(A) / P(B)

| Part | Name | What it means |
|------|------|---------------|
| P(A\|B) | Posterior | Updated belief about A after seeing evidence B |
| P(B\|A) | Likelihood | How probable the evidence B is if A is true |
| P(A) | Prior | Your belief about A before seeing evidence |
| P(B) | Evidence | Total probability of seeing B |
```


# The Disease Testing Example

---

A disease affects 1 in 10,000 people. Test is 99% accurate (catches 99% of sick, 1% false positive).

```
P(sick)          = 0.0001
P(positive|sick) = 0.99
P(positive|healthy) = 0.01

P(positive) = 0.99 × 0.0001 + 0.01 × 0.9999 = 0.010098
P(sick|positive) = 0.99 × 0.0001 / 0.010098 = 0.0098 ≈ 0.98%
```

Less than 1%! The prior dominates. When a condition is rare, even accurate tests produce mostly false positives.


In [ ]:
def bayes(prior, likelihood, fpr):    evidence = likelihood * prior + fpr * (1 - prior)    return likelihood * prior / evidence, evidencepost, ev = bayes(0.0001, 0.99, 0.01)print(f"P(sick|positive) = {post:.6f} ({post*100:.2f}%)")print(f"P(positive) = {ev:.6f}")

In [ ]:
import numpy as npimport matplotlib.pyplot as plt# Beta-Binomial updating: start uniform Beta(1,1)import mathfrom math import lgammadef beta_pdf(x, a, b):    return math.exp(lgamma(a + b) - lgamma(a) - lgamma(b) + (a - 1) * math.log(x) + (b - 1) * math.log(1 - x))x = np.linspace(0.001, 0.999, 200)plt.plot(x, [beta_pdf(v, 1, 1) for v in x], label='Prior Beta(1,1)', color='#95a5a6')plt.plot(x, [beta_pdf(v, 8, 4) for v in x], label='Posterior Beta(8,4)', color='#e74c3c', lw=2)plt.plot(x, [beta_pdf(v, 13, 9) for v in x], label='Posterior Beta(13,9)', color='#27ae60', lw=2)plt.title('Bayesian Updating: 7H3T then 5H5T', fontsize=12, fontweight='bold')plt.xlabel('Probability of heads'); plt.ylabel('Density')plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()print(f"Prior mean: 1/(1+1) = 0.5")print(f"Posterior 1 mean: 8/(8+4) = 0.667")print(f"Posterior 2 mean: 13/(13+9) = 0.591")

# Naive Bayes: Independence Assumption

---

Naive Bayes assumes features are conditionally independent given the class:

```
score(class) = P(class) × product of P(feature_i | class)
```

The denominator is the same for all classes, so we compare numerators and pick the highest.

Laplace smoothing prevents zero probabilities:
```
P(word|class) = (count(word, class) + 1) / (total_words_in_class + vocab_size)
```


In [ ]:
import mathfrom collections import defaultdictclass NaiveBayes:    def __init__(self, smoothing=1.0):        self.smoothing = smoothing        self.class_counts = defaultdict(int)        self.word_counts = defaultdict(lambda: defaultdict(int))        self.class_word_totals = defaultdict(int)        self.vocab = set()    def train(self, documents, labels):        for doc, label in zip(documents, labels):            self.class_counts[label] += 1            words = doc.lower().split()            for word in words:                self.word_counts[label][word] += 1                self.class_word_totals[label] += 1                self.vocab.add(word)    def predict(self, document):        words = document.lower().split()        total_docs = sum(self.class_counts.values())        vocab_size = len(self.vocab)        best_class = None; best_score = float('-inf')        for cls in self.class_counts:            score = math.log(self.class_counts[cls] / total_docs)            for word in words:                count = self.word_counts[cls].get(word, 0)                total = self.class_word_totals[cls]                score += math.log((count + self.smoothing) / (total + self.smoothing * vocab_size))            if score > best_score:                best_score = score; best_class = cls        return best_class# Train on spam datatrain_docs = [    "win free money now", "free lottery ticket winner", "claim your prize today free",    "urgent offer free cash", "congratulations you won free",    "meeting tomorrow at noon", "project update attached",    "can we schedule a call", "quarterly report review",    "lunch on thursday sounds good", "team standup notes attached",    "please review the pull request",]train_labels = ["spam"]*5 + ["ham"]*7nb = NaiveBayes()nb.train(train_docs, train_labels)test_messages = [    "free money waiting for you", "meeting rescheduled to friday",    "you won a free prize", "please review the attached report",]for msg in test_messages:    print(f"  '{msg}' -> {nb.predict(msg)}")

In [ ]:
import numpy as npimport matplotlib.pyplot as plt# Show top spam vs ham wordsvocab_size = len(nb.vocab)spam_probs = {}ham_probs = {}for word in nb.vocab:    spam_sc = nb.word_counts['spam'].get(word, 0)    ham_sc = nb.word_counts['ham'].get(word, 0)    spam_t = nb.class_word_totals['spam']    ham_t = nb.class_word_totals['ham']    spam_probs[word] = (spam_sc + 1) / (spam_t + vocab_size)    ham_probs[word] = (ham_sc + 1) / (ham_t + vocab_size)spam_top = sorted(spam_probs.items(), key=lambda x: x[1], reverse=True)[:6]ham_top = sorted(ham_probs.items(), key=lambda x: x[1], reverse=True)[:6]fig, axes = plt.subplots(1, 2, figsize=(10, 4))axes[0].barh([w for w, _ in spam_top][::-1], [p for _, p in spam_top][::-1], color='#e74c3c')axes[0].set_title('Top Spam Words', fontweight='bold')axes[1].barh([w for w, _ in ham_top][::-1], [p for _, p in ham_top][::-1], color='#27ae60')axes[1].set_title('Top Ham Words', fontweight='bold')for ax in axes: ax.grid(True, alpha=0.3, axis='x')plt.tight_layout(); plt.show()

# Quiz Review

---

Try answering these before reading the explanations:

**Q1:** Base rate fallacy?
> **Explanation:** Ignoring the prior when interpreting evidence — a rare disease means even accurate tests give low posterior.

**Q2:** Why log probabilities in Naive Bayes?
> **Explanation:** Multiplying many small probabilities underflows to zero; summing log-probs is numerically stable.

**Q3:** Laplace smoothing?
> **Explanation:** Adds a small count to every feature so no probability is ever zero.



# Practice Exercises

---

Do these in order:

**Exercise 1:** Run the spam classifier with smoothing values 0.01, 0.1, 1.0, 10.0 and observe changes.

**Exercise 2:** Multiple tests: a patient tests positive twice (99% accurate, prevalence 1/10000). What is P(sick)?



# Key Terms

---

| Term | What people say | What it actually means |
|------|----------------|----------------------|
| Prior | My initial guess | P(hypothesis) before seeing evidence |
| Likelihood | How well the data fits | P(evidence|hypothesis) |
| Posterior | My updated belief | P(hypothesis|evidence) |
| Naive Bayes | Simple text classifier | Assumes features independent given the class |
| Laplace smoothing | Add-one smoothing | Prevents zero probabilities from unseen data |
| MLE | Just use frequencies | Maximize P(data|parameters), no prior |
| MAP | MLE with a prior | Maximize P(data|params) × P(params) |


# Further Reading

---

- [3Blue1Brown: Bayes' theorem](https://www.youtube.com/watch?v=HZGCoVF3YvM)
- [Think Bayes](https://greenteapress.com/wp/think-bayes/) - free book, Bayesian statistics with Python code


# Next Steps

---

When you're ready, continue to Lesson 08: Optimization.

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
